# Modul 12: Unüberwachtes Lernen und Textklassifikation

    **Notebooktyp:** Übungs- und Bewertungsnotebook  
    **Vorlesungen dieses Moduls:** Unüberwacht lernen, Text als Merkmale  
    **Erwarteter Schwierigkeitsgrad:** Fortgeschritten  
    **Orientierungszeit:** etwa 130 bis 180 Minuten

    ## Überblick

    Sie vergleichen mehrere Cluster- und Anomalieverfahren, nutzen wenige Labels semi-supervised und bauen anschließend vollständig lokale Textmerkmale und Textklassifikationspipelines auf.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_12A_20260723.ipynb`
- `ML Für Anfänger - Record_Module_12B_20260823.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - KMeans, MiniBatchKMeans, hierarchisches Clustering, DBSCAN, OPTICS und GaussianMixture vergleichen.
- IsolationForest, LocalOutlierFactor und OneClassSVM auf skalierten Daten anwenden.
- Semi-supervised Verfahren mit wenigen Labels untersuchen und Unsicherheit dokumentieren.
- Kleine lokale Textsammlungen mit Labels strukturieren und bereinigen.
- Count-, TF-IDF- und Hashing-Vektorisierung erzeugen.
- Einfache Textklassifikationspipelines trainieren, validieren, analysieren und speichern.

    ## Bewertete Fähigkeiten

    - Clusterlabels, Rauschen, Silhouette und Modellannahmen
- Anomaliescores und Novelty Detection
- LabelPropagation und LabelSpreading mit maskierten Labels
- CountVectorizer, TfidfVectorizer, HashingVectorizer und Sparse-Matrizen
- MultinomialNB, LogisticRegression, Fehleranalyse und Serialisierung

## Arbeitsanweisungen

Bearbeiten Sie die Aufgaben in der angegebenen Reihenfolge. Schreiben Sie Ihren Code ausschließlich in die klar markierten Arbeitszellen. Ergänzen Sie nach jeder Aufgabe eine kurze fachliche Reflexion. Verwenden Sie das Testset nicht für Modellwahl oder Hyperparameterentscheidungen, sofern die Aufgabe dies nicht ausdrücklich als abschließenden Schritt verlangt.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# Gemeinsames Setup für dieses Notebook
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import io
import joblib
import re
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans, MiniBatchKMeans, OPTICS
from sklearn.datasets import make_blobs, make_moons
from sklearn.ensemble import IsolationForest
from sklearn.feature_extraction.text import CountVectorizer, HashingVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, silhouette_score
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import LocalOutlierFactor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.semi_supervised import LabelPropagation, LabelSpreading
from sklearn.svm import OneClassSVM
from sklearn.linear_model import LogisticRegression

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)
np.set_printoptions(precision=3, suppress=True)
pd.set_option("display.max_columns", 50)
warnings.filterwarnings("ignore", category=FutureWarning)

X_cluster_12, hidden_cluster_12 = make_blobs(
    n_samples=330,
    centers=[(-4, -2), (0, 4), (4, -1)],
    cluster_std=[0.55, 1.35, 0.75],
    random_state=RANDOM_SEED,
)
X_cluster_12 = np.vstack(
    [X_cluster_12, np.array([[9, 9], [-9, 7], [8, -8], [-8, -7]], dtype=float)]
)

X_ssl_12, y_ssl_12 = make_moons(
    n_samples=300,
    noise=0.12,
    random_state=RANDOM_SEED,
)

text_samples_12 = [
    ("Die Rechnung enthält einen falschen Betrag", "abrechnung"),
    ("Meine Kreditkarte wurde doppelt belastet", "abrechnung"),
    ("Bitte erklären Sie die monatliche Gebühr", "abrechnung"),
    ("Ich brauche eine korrigierte Rechnung", "abrechnung"),
    ("Die Erstattung ist noch nicht angekommen", "abrechnung"),
    ("Der Rabatt fehlt auf meiner Rechnung", "abrechnung"),
    ("Mein Login funktioniert seit heute nicht", "technik"),
    ("Die App stürzt beim Start sofort ab", "technik"),
    ("Ich kann mein Passwort nicht zurücksetzen", "technik"),
    ("Die Verbindung zum Server bricht ab", "technik"),
    ("Nach dem Update bleibt der Bildschirm leer", "technik"),
    ("Der Download endet immer mit einem Fehler", "technik"),
    ("Wie kann ich mein Paket verfolgen", "versand"),
    ("Die Lieferung ist mehrere Tage verspätet", "versand"),
    ("Das Paket wurde an die falsche Adresse geschickt", "versand"),
    ("Wann wird meine Bestellung versendet", "versand"),
    ("Der Zustellstatus hat sich nicht verändert", "versand"),
    ("Ich möchte den Liefertermin ändern", "versand"),
    ("Bitte ändern Sie meine hinterlegte E-Mail-Adresse", "konto"),
    ("Ich möchte mein Kundenkonto schließen", "konto"),
    ("Wo kann ich meine Profildaten bearbeiten", "konto"),
    ("Meine Telefonnummer im Konto ist falsch", "konto"),
    ("Ich brauche eine Kopie meiner gespeicherten Daten", "konto"),
    ("Wie aktiviere ich die Zwei-Faktor-Anmeldung", "konto"),
]
text_data_12 = pd.DataFrame(text_samples_12, columns=["text", "label"])

print("Setup abgeschlossen. Zufallsstartwert:", RANDOM_SEED)


## Aufgabe 1: Clusterverfahren und Silhouette vergleichen

    Skalieren Sie `X_cluster_12` und vergleichen Sie:

- KMeans,
- MiniBatchKMeans,
- AgglomerativeClustering,
- DBSCAN,
- OPTICS,
- GaussianMixture.

Berechnen Sie Anzahl gefundener Cluster, Anteil als Rauschen markierter Punkte und, sofern möglich, Silhouette Score. Visualisieren Sie jede Zuordnung in einer eigenen Abbildung.

> **Hinweis:** Entfernen Sie Rauschlabel -1 nur für die Silhouette-Berechnung, nicht aus der Dokumentation.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 1

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Entfernen Sie Rauschlabel -1 nur für die Silhouette-Berechnung, nicht aus der Dokumentation.

## Aufgabe 2: Anomalieverfahren auf skalierten Daten vergleichen

    Verwenden Sie die skalierten Clusterdaten und vergleichen Sie:

- `IsolationForest`,
- `LocalOutlierFactor`,
- `OneClassSVM`.

Markieren Sie jeweils ungefähr vier Prozent der Beobachtungen als ungewöhnlich, soweit das Verfahren dies erlaubt. Erstellen Sie eine Tabelle mit Anomalielabeln und Scores, zählen Sie Übereinstimmungen und visualisieren Sie Punkte, die von mindestens zwei Verfahren markiert werden.

> **Hinweis:** Vergleichen Sie zunächst Labels oder Ränge, nicht rohe Scores unterschiedlicher Modelle.

In [ ]:
# Nutzen Sie `X_scaled` aus Aufgabe 1 oder skalieren Sie erneut.

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 2

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Vergleichen Sie zunächst Labels oder Ränge, nicht rohe Scores unterschiedlicher Modelle.

## Aufgabe 3: Semi-supervised Lernen mit wenigen Labels

    Teilen Sie `X_ssl_12`, `y_ssl_12` stratifiziert in Training und Test. Behalten Sie im Training nur drei bekannte Labels je Klasse und setzen Sie alle übrigen Trainingslabels auf `-1`.

1. Trainieren Sie `LabelPropagation` und `LabelSpreading`.
2. Bewerten Sie beide auf den vollständig gelabelten Testdaten.
3. Geben Sie vorhergesagte Klassenwahrscheinlichkeiten beziehungsweise Labelverteilungen für fünf ursprünglich ungelabelte Trainingspunkte aus.
4. Variieren Sie `gamma` oder `alpha` einmal und dokumentieren Sie die Empfindlichkeit.

> **Hinweis:** Die wahren Labels der maskierten Trainingspunkte dürfen nur zur nachträglichen Bewertung dienen.

In [ ]:
X_train_ssl, X_test_ssl, y_train_ssl, y_test_ssl = train_test_split(
    X_ssl_12, y_ssl_12, test_size=0.30,
    random_state=RANDOM_SEED, stratify=y_ssl_12
)

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 3

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Die wahren Labels der maskierten Trainingspunkte dürfen nur zur nachträglichen Bewertung dienen.

## Aufgabe 4: Text bereinigen und drei Vektorisierungen untersuchen

    Bereinigen Sie `text_data_12["text"]` mit einer kleinen Funktion, die Kleinschreibung, Leerzeichenbereinigung und das Entfernen nicht alphabetischer Zeichen durchführt.

Vergleichen Sie:

- `CountVectorizer` mit Uni- und Bigrammen,
- `TfidfVectorizer` mit Uni- und Bigrammen,
- `HashingVectorizer(n_features=32, alternate_sign=False)`.

Geben Sie Matrixformen, Anzahl Nichtnullwerte, Vokabularbeispiele und die fünf höchsten TF-IDF-Gewichte des ersten Dokuments aus.

> **Hinweis:** Rufen Sie bei großen Textdaten nicht unkritisch `.toarray()` auf die gesamte Matrix auf.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 4

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Rufen Sie bei großen Textdaten nicht unkritisch `.toarray()` auf die gesamte Matrix auf.

## Aufgabe 5: Integrationsaufgabe: Textpipelines vergleichen und speichern

    Teilen Sie die lokale Textsammlung stratifiziert in Training und Test. Vergleichen Sie:

1. CountVectorizer + MultinomialNB,
2. TfidfVectorizer + LogisticRegression.

Berechnen Sie Genauigkeit und Klassifikationsbericht, erstellen Sie eine Fehleranalyse mit Originaltext, wahrer und vorhergesagter Klasse und speichern/laden Sie die bessere Pipeline über einen `BytesIO`-Puffer mit `joblib`. Prüfen Sie, dass Vorhersagen vor und nach dem Laden identisch sind.

> **Hinweis:** Serialisieren Sie Vektorisierer und Modell gemeinsam als Pipeline.

In [ ]:
X_text_train, X_text_test, y_text_train, y_text_test = train_test_split(
    text_data_12["text"], text_data_12["label"], test_size=0.33,
    random_state=RANDOM_SEED, stratify=text_data_12["label"]
)

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 5

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Serialisieren Sie Vektorisierer und Modell gemeinsam als Pipeline.

## Abschluss und Selbstkontrolle

Prüfen Sie vor der Abgabe, ob alle Arbeitszellen ausgefüllt sind, das Notebook von oben nach unten ohne unerwartete Fehler läuft, alle Diagramme beschriftet sind und jede Reflexion Ihre Beobachtungen sowie mindestens eine mögliche Fehlerquelle enthält.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.